# Grad-CAM for Financial Markets

This notebook demonstrates how to use Gradient-weighted Class Activation Mapping (Grad-CAM)
to interpret CNN-based trading signals. We'll:

1. Fetch cryptocurrency data from Bybit
2. Train a CNN to predict price movements
3. Use Grad-CAM to visualize which time periods influence predictions
4. Run a backtest with interpretable signals

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import torch
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from model import FinancialCNN, GradCAM, GradCAMPlusPlus, visualize_gradcam
from train import (
    fetch_bybit_data, fetch_yahoo_data, 
    prepare_sequences, create_labels,
    generate_synthetic_data, train_model,
    FinancialDataset
)
from backtest import BacktestEngine, PerformanceMetrics

from torch.utils.data import DataLoader

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 1. Data Loading

We'll try to fetch real data from Bybit. If that fails, we'll use synthetic data.

In [ ]:
# Configuration
SYMBOL = "BTCUSDT"
SEQUENCE_LENGTH = 60
BATCH_SIZE = 32

# Try to fetch real data
print(f"Fetching {SYMBOL} data from Bybit...")
ohlcv_data = fetch_bybit_data(symbol=SYMBOL, interval="60", limit=1000)

if ohlcv_data is not None:
    print(f"Fetched {len(ohlcv_data)} candles")
    print(f"Data shape: {ohlcv_data.shape}")
    print(f"Price range: ${ohlcv_data[:, 3].min():,.2f} - ${ohlcv_data[:, 3].max():,.2f}")
    use_real_data = True
else:
    print("Failed to fetch real data. Using synthetic data.")
    use_real_data = False

In [ ]:
# Prepare data for training
if use_real_data:
    # Create labels based on future price movement
    labels = create_labels(ohlcv_data[:, 3], threshold=0.005, lookahead=1)
    features, valid_labels = prepare_sequences(ohlcv_data, labels, SEQUENCE_LENGTH)
else:
    # Generate synthetic data
    features, valid_labels = generate_synthetic_data(
        num_samples=2000,
        sequence_length=SEQUENCE_LENGTH
    )

print(f"Features shape: {features.shape}")
print(f"Labels shape: {valid_labels.shape}")
print(f"Label distribution: {np.bincount(valid_labels)}")
print(f"  Down: {(valid_labels == 0).sum()}")
print(f"  Neutral: {(valid_labels == 1).sum()}")
print(f"  Up: {(valid_labels == 2).sum()}")

## 2. Model Training

We'll train a 1D CNN to predict price movement direction.

In [ ]:
# Split data
n_samples = len(features)
train_size = int(0.7 * n_samples)
val_size = int(0.15 * n_samples)

train_features = features[:train_size]
train_labels = valid_labels[:train_size]
val_features = features[train_size:train_size + val_size]
val_labels = valid_labels[train_size:train_size + val_size]
test_features = features[train_size + val_size:]
test_labels = valid_labels[train_size + val_size:]

print(f"Train samples: {len(train_features)}")
print(f"Validation samples: {len(val_features)}")
print(f"Test samples: {len(test_features)}")

In [ ]:
# Create data loaders
train_dataset = FinancialDataset(train_features, train_labels)
val_dataset = FinancialDataset(val_features, val_labels)
test_dataset = FinancialDataset(test_features, test_labels)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [ ]:
# Create model
model = FinancialCNN(
    input_channels=5,
    num_classes=3,
    sequence_length=SEQUENCE_LENGTH,
    hidden_dims=[32, 64, 128],
    dropout=0.3
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(model)

In [ ]:
# Train the model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on {device}...")

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=30,
    learning_rate=0.001,
    device=device,
    patience=10
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy plot
axes[1].plot(history['train_acc'], label='Train Accuracy')
axes[1].plot(history['val_acc'], label='Val Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 3. Grad-CAM Visualization

Now let's use Grad-CAM to understand which time periods the model focuses on.

In [ ]:
# Create Grad-CAM object
model.eval()
model = model.to('cpu')  # Move to CPU for visualization
gradcam = GradCAM(model)
gradcam_pp = GradCAMPlusPlus(model)

In [ ]:
# Visualize Grad-CAM for a few test samples
class_names = ['Down', 'Neutral', 'Up']

fig, axes = plt.subplots(3, 2, figsize=(16, 12))

for i in range(3):
    # Get a sample
    idx = np.random.randint(len(test_features))
    sample = test_features[idx]
    true_label = test_labels[idx]
    
    # Create input tensor
    input_tensor = torch.FloatTensor(sample).unsqueeze(0)
    
    # Get prediction and confidence
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        pred_class = probs.argmax(dim=1).item()
        confidence = probs[0, pred_class].item()
    
    # Get Grad-CAM heatmap
    input_tensor = torch.FloatTensor(sample).unsqueeze(0)
    input_tensor.requires_grad_(True)
    cam = gradcam(input_tensor)
    
    # Plot price with CAM overlay
    ax1 = axes[i, 0]
    close_prices = sample[3]  # Close prices (normalized)
    x = np.arange(len(close_prices))
    
    # Color by importance
    scatter = ax1.scatter(x, close_prices, c=cam, cmap='RdYlBu_r', s=30)
    ax1.plot(x, close_prices, 'k-', alpha=0.3)
    plt.colorbar(scatter, ax=ax1, label='Importance')
    
    ax1.set_title(f'Sample {i+1}: True={class_names[true_label]}, Pred={class_names[pred_class]} ({confidence:.2%})')
    ax1.set_xlabel('Time Period')
    ax1.set_ylabel('Normalized Price')
    
    # Plot CAM heatmap
    ax2 = axes[i, 1]
    ax2.bar(x, cam, color=plt.cm.RdYlBu_r(cam))
    ax2.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Threshold')
    ax2.set_title('Grad-CAM Attention')
    ax2.set_xlabel('Time Period')
    ax2.set_ylabel('Attention Weight')
    ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Compare Grad-CAM and Grad-CAM++
idx = np.random.randint(len(test_features))
sample = test_features[idx]

input_tensor1 = torch.FloatTensor(sample).unsqueeze(0)
input_tensor1.requires_grad_(True)
cam1 = gradcam(input_tensor1)

input_tensor2 = torch.FloatTensor(sample).unsqueeze(0)
input_tensor2.requires_grad_(True)
cam2 = gradcam_pp(input_tensor2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(len(cam1)), cam1, color='steelblue')
axes[0].set_title('Grad-CAM')
axes[0].set_xlabel('Time Period')
axes[0].set_ylabel('Attention')

axes[1].bar(range(len(cam2)), cam2, color='darkorange')
axes[1].set_title('Grad-CAM++')
axes[1].set_xlabel('Time Period')
axes[1].set_ylabel('Attention')

plt.tight_layout()
plt.show()

## 4. Attention Analysis

Let's analyze where the model typically focuses its attention.

In [ ]:
# Compute average attention across many samples
n_samples_to_analyze = min(100, len(test_features))
all_cams = []

for i in range(n_samples_to_analyze):
    sample = test_features[i]
    input_tensor = torch.FloatTensor(sample).unsqueeze(0)
    input_tensor.requires_grad_(True)
    cam = gradcam(input_tensor)
    all_cams.append(cam)

all_cams = np.array(all_cams)
mean_cam = all_cams.mean(axis=0)
std_cam = all_cams.std(axis=0)

# Plot average attention
plt.figure(figsize=(12, 5))
x = np.arange(len(mean_cam))
plt.fill_between(x, mean_cam - std_cam, mean_cam + std_cam, alpha=0.3)
plt.plot(x, mean_cam, 'b-', linewidth=2)
plt.xlabel('Time Period')
plt.ylabel('Average Attention')
plt.title(f'Average Grad-CAM Attention Across {n_samples_to_analyze} Samples')
plt.grid(True, alpha=0.3)
plt.show()

# Print statistics
print(f"Most attended period: {mean_cam.argmax()} (attention: {mean_cam.max():.3f})")
print(f"Least attended period: {mean_cam.argmin()} (attention: {mean_cam.min():.3f})")
print(f"Recent periods (last 10) average attention: {mean_cam[-10:].mean():.3f}")
print(f"Historical periods (first 10) average attention: {mean_cam[:10].mean():.3f}")

## 5. Backtesting with Interpretable Signals

Finally, let's run a backtest and analyze the trading decisions.

In [ ]:
# Generate OHLCV data for backtesting
if use_real_data:
    backtest_data = ohlcv_data
else:
    # Create synthetic OHLCV data
    np.random.seed(42)
    n_periods = 1000
    base_price = 100
    
    backtest_data = []
    for i in range(n_periods):
        trend = np.sin(i / 50) * 0.002  # Cyclical trend
        noise = np.random.randn() * 0.01
        price = base_price * (1 + trend + noise)
        volatility = 0.01
        
        open_p = price * (1 + np.random.randn() * volatility * 0.5)
        high_p = max(open_p, price) * (1 + abs(np.random.randn()) * volatility)
        low_p = min(open_p, price) * (1 - abs(np.random.randn()) * volatility)
        volume = np.random.uniform(1000, 10000)
        
        backtest_data.append([open_p, high_p, low_p, price, volume])
        base_price = price
    
    backtest_data = np.array(backtest_data)

print(f"Backtest data shape: {backtest_data.shape}")

In [ ]:
# Create backtest engine
engine = BacktestEngine(
    model=model,
    sequence_length=SEQUENCE_LENGTH,
    confidence_threshold=0.5,
    cam_threshold=0.5
)

# Run backtest
result = engine.run(
    data=backtest_data,
    initial_capital=100000,
    position_size=0.1,
    transaction_cost=0.001
)

print("Backtest Results:")
print("=" * 50)
for key, value in result.metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Plot equity curve
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Equity curve
axes[0].plot(result.equity_curve, 'b-', linewidth=1.5)
axes[0].axhline(y=100000, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Time Period')
axes[0].set_ylabel('Portfolio Value ($)')
axes[0].set_title('Equity Curve')
axes[0].grid(True, alpha=0.3)

# Mark trades
for trade in result.trades[:50]:  # Show first 50 trades
    color = 'green' if trade.pnl > 0 else 'red'
    axes[0].axvline(x=trade.entry_time, color=color, alpha=0.3, linewidth=0.5)

# Drawdown
peak = np.maximum.accumulate(result.equity_curve)
drawdown = (peak - result.equity_curve) / peak * 100
axes[1].fill_between(range(len(drawdown)), drawdown, alpha=0.5, color='red')
axes[1].set_xlabel('Time Period')
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_title('Drawdown')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze trade explanations
if len(result.trades) > 0:
    print("Sample Trade Explanations:")
    print("=" * 50)
    
    for i, trade in enumerate(result.trades[:5]):
        explanation = engine.explain_trade(trade, backtest_data)
        print(f"\nTrade {i+1}:")
        print(f"  Direction: {explanation['direction']}")
        print(f"  Entry Price: ${explanation['entry_price']:.2f}")
        print(f"  Exit Price: ${explanation['exit_price']:.2f}")
        print(f"  PnL: {explanation['pnl']*100:.2f}%")
        print(f"  Confidence: {explanation['confidence']:.2%}")
        print(f"  Important Periods: {explanation['num_important_periods']}")
        if 'focus_ratio' in explanation:
            print(f"  Recent/Historical Focus Ratio: {explanation['focus_ratio']:.2f}")

In [ ]:
# Visualize a specific trade with its Grad-CAM explanation
if len(result.trades) > 0 and result.trades[0].gradcam_explanation is not None:
    trade = result.trades[0]
    window = backtest_data[trade.entry_time - SEQUENCE_LENGTH:trade.entry_time]
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    
    # Price chart
    x = np.arange(len(window))
    close_prices = window[:, 3]
    
    # Color by importance
    cam = trade.gradcam_explanation
    for i in range(len(close_prices) - 1):
        color = plt.cm.RdYlBu_r(cam[i])
        axes[0].plot([x[i], x[i+1]], [close_prices[i], close_prices[i+1]], 
                     color=color, linewidth=2)
    
    axes[0].set_title(f"Trade: {'LONG' if trade.direction == 1 else 'SHORT'} | "
                      f"PnL: {trade.pnl*100:.2f}% | Confidence: {trade.confidence:.2%}")
    axes[0].set_ylabel('Price')
    
    # CAM heatmap
    axes[1].imshow(cam.reshape(1, -1), aspect='auto', cmap='RdYlBu_r')
    axes[1].set_ylabel('CAM')
    axes[1].set_yticks([])
    
    # Volume
    colors = [plt.cm.RdYlBu_r(c) for c in cam]
    axes[2].bar(x, window[:, 4], color=colors, alpha=0.7)
    axes[2].set_ylabel('Volume')
    axes[2].set_xlabel('Time Period')
    
    plt.tight_layout()
    plt.show()

## Summary

In this notebook, we demonstrated:

1. **Data Loading**: Fetching OHLCV data from Bybit (or using synthetic data)
2. **Model Training**: Training a 1D CNN for price movement prediction
3. **Grad-CAM Visualization**: Using Grad-CAM to understand which time periods influence predictions
4. **Backtesting**: Running a trading strategy with interpretable signals

Key findings:
- Grad-CAM reveals which parts of the price history the model focuses on
- Different predictions (up/down/neutral) may focus on different patterns
- The model's attention can help identify potential issues (e.g., if it ignores recent data)
- Trade explanations can help build trust in AI-driven trading decisions